# Personal Finance Manager

Run the cells **top to bottom**. Cells 2 to 8 write the project files into one folder; the last cell starts the app.

Every module goes in the **same folder** so the imports (`from expense import ...`) work.

In [11]:
import os, shutil

# Start clean (safe even if the folder does not exist yet)
shutil.rmtree("personal_finance_manager", ignore_errors=True)
os.makedirs("personal_finance_manager/data", exist_ok=True)
os.makedirs("personal_finance_manager/backups", exist_ok=True)
os.makedirs("personal_finance_manager/reports", exist_ok=True)

## expense.py

In [12]:
%%writefile "personal_finance_manager/expense.py"
"""
expense.py
----------
Defines the Expense class: the core data model of the Personal Finance
Manager. Dates are stored as YYYY-MM-DD strings everywhere, which keeps
the CSV readable and makes plain string sorting chronological.
"""

import itertools
from datetime import datetime

DATE_FORMAT = "%Y-%m-%d"
CATEGORIES = [
    "Food",
    "Transport",
    "Entertainment",
    "Shopping",
    "Bills",
    "Health",
    "Education",
    "Other",
]


class ExpenseValidationError(ValueError):
    """Raised when an Expense is built (or changed) with invalid data."""


def validate_amount(amount):
    try:
        amount = float(amount)
    except (TypeError, ValueError):
        raise ExpenseValidationError(f"Amount must be a number, got: {amount!r}")
    if amount != amount or amount in (float("inf"), float("-inf")):
        raise ExpenseValidationError("Amount must be a finite number.")
    if amount <= 0:
        raise ExpenseValidationError("Amount must be greater than zero.")
    return round(amount, 2)


def validate_category(category):
    if category is None or not str(category).strip():
        raise ExpenseValidationError("Category cannot be empty.")
    return str(category).strip().title()


def validate_date(date):
    try:
        datetime.strptime(str(date), DATE_FORMAT)
    except ValueError:
        raise ExpenseValidationError(f"Date must be in YYYY-MM-DD format, got: {date!r}")
    return str(date)


class Expense:
    """A single expense record: amount, category, date, description."""

    _id_counter = itertools.count(1)

    def __init__(self, amount, category, date, description="", expense_id=None):
        self.amount = validate_amount(amount)
        self.category = validate_category(category)
        self.date = validate_date(date)
        self.description = str(description).strip() if description else ""
        self.id = expense_id if expense_id is not None else next(Expense._id_counter)

    @classmethod
    def reset_id_counter(cls, start):
        cls._id_counter = itertools.count(start)

    @property
    def date_obj(self):
        return datetime.strptime(self.date, DATE_FORMAT)

    def update(self, amount=None, category=None, date=None, description=None):
        """Change fields with the same validation as the constructor.
        Nothing is modified unless every supplied value is valid."""
        new_amount = validate_amount(amount) if amount is not None else self.amount
        new_category = validate_category(category) if category is not None else self.category
        new_date = validate_date(date) if date is not None else self.date
        new_desc = str(description).strip() if description is not None else self.description
        self.amount, self.category, self.date, self.description = (
            new_amount, new_category, new_date, new_desc,
        )

    def to_row(self):
        """CSV row: [ID, Date, Category, Amount, Description]."""
        return [self.id, self.date, self.category, f"{self.amount:.2f}", self.description]

    @classmethod
    def from_row(cls, row):
        expense_id, date, category, amount, description = (list(row) + [""] * 5)[:5]
        return cls(
            amount=amount,
            category=category,
            date=date,
            description=description,
            expense_id=int(expense_id) if str(expense_id).strip() else None,
        )

    def __str__(self):
        desc = f" - {self.description}" if self.description else ""
        return f"{self.date} | {self.category}: ₹{self.amount:,.2f}{desc}"

    def __repr__(self):
        return (f"Expense(id={self.id}, amount={self.amount}, category={self.category!r}, "
                f"date={self.date!r}, description={self.description!r})")

    def __lt__(self, other):
        return (self.date, self.id) < (other.date, other.id)


Writing personal_finance_manager/expense.py


## utils.py

In [13]:
%%writefile "personal_finance_manager/utils.py"
"""
utils.py
--------
Validation, parsing and console-formatting helpers shared by the app.
"""

from datetime import datetime, timedelta

from expense import CATEGORIES, DATE_FORMAT, ExpenseValidationError


class Color:
    CYAN = "\033[96m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    BOLD = "\033[1m"
    END = "\033[0m"


def colorize(text, color):
    return f"{color}{text}{Color.END}"


def print_success(msg):
    print(colorize(f"✅ {msg}", Color.GREEN))


def print_error(msg):
    print(colorize(f"❌ {msg}", Color.RED))


def print_warning(msg):
    print(colorize(f"⚠ {msg}", Color.YELLOW))


def format_currency(value):
    try:
        return f"₹{float(value):,.2f}"
    except (TypeError, ValueError):
        return "₹0.00"


# ----------------------------------------------------------------------
# Parsers: return a clean value or raise ValueError with a friendly message
# ----------------------------------------------------------------------

def parse_amount(raw):
    raw = raw.strip().replace(",", "").replace("₹", "")
    if not raw:
        raise ValueError("Amount cannot be empty.")
    try:
        value = float(raw)
    except ValueError:
        raise ValueError(f"'{raw}' is not a valid number.")
    if value != value or value in (float("inf"), float("-inf")):
        raise ValueError("Amount must be a finite number.")
    if value <= 0:
        raise ValueError("Amount must be greater than zero.")
    return round(value, 2)


def parse_date(raw, allow_empty_today=True):
    """Accept YYYY-MM-DD, or blank/'today'/'yesterday' as shortcuts."""
    raw = raw.strip().lower()
    if raw == "today" or (raw == "" and allow_empty_today):
        return datetime.now().strftime(DATE_FORMAT)
    if raw == "yesterday":
        return (datetime.now() - timedelta(days=1)).strftime(DATE_FORMAT)
    try:
        parsed = datetime.strptime(raw, DATE_FORMAT)
    except ValueError:
        raise ValueError(f"'{raw}' is not a valid date. Use YYYY-MM-DD (e.g. 2024-01-15).")
    if parsed > datetime.now() + timedelta(days=1):
        raise ValueError("Date cannot be in the future.")
    return parsed.strftime(DATE_FORMAT)


def parse_category(raw):
    """Any non-empty category is accepted (so users can add their own);
    it is title-cased so 'food' and 'FOOD' end up the same."""
    raw = raw.strip()
    if not raw:
        raise ValueError("Category cannot be empty.")
    if len(raw) > 40:
        raise ValueError("Category is too long (max 40 characters).")
    return raw.title()


def parse_description(raw, max_len=200):
    raw = raw.strip()
    if len(raw) > max_len:
        raise ValueError(f"Description is too long (max {max_len} characters).")
    return raw


# ----------------------------------------------------------------------
# Prompt helpers
# ----------------------------------------------------------------------

def prompt_with_validation(prompt_text, parser, default=None, allow_cancel=True):
    """Retry until `parser(raw)` succeeds. Typing 'c' cancels (returns None).
    If `default` is given, pressing Enter on a blank line returns it."""
    while True:
        raw = input(prompt_text)
        if allow_cancel and raw.strip().lower() in ("c", "cancel"):
            return None
        if raw.strip() == "" and default is not None:
            return default
        try:
            return parser(raw)
        except (ValueError, ExpenseValidationError) as exc:
            print_error(str(exc))


def prompt_nonempty(prompt_text):
    while True:
        value = input(prompt_text).strip()
        if value:
            return value
        print_error("This field cannot be empty. Please try again.")


def prompt_int_in_range(prompt_text, low, high):
    while True:
        raw = input(prompt_text).strip()
        try:
            value = int(raw)
        except ValueError:
            print_error("Please enter a valid number.")
            continue
        if not (low <= value <= high):
            print_error(f"Please enter a number between {low} and {high}.")
            continue
        return value


def confirm(prompt_text="Are you sure? (y/n): "):
    while True:
        raw = input(prompt_text).strip().lower()
        if raw in ("y", "yes"):
            return True
        if raw in ("n", "no"):
            return False
        print_error("Please enter 'y' or 'n'.")


def category_hint():
    return "/".join(CATEGORIES)


Writing personal_finance_manager/utils.py


## file_manager.py

In [14]:
%%writefile "personal_finance_manager/file_manager.py"
"""
file_manager.py
---------------
All disk access: CSV load/save, timestamped backups, restore, and exports.

CSV columns: ID, Date, Category, Amount, Description
"""

import csv
import os
import shutil
from datetime import datetime

from expense import Expense, ExpenseValidationError

HEADER = ["ID", "Date", "Category", "Amount", "Description"]


class FileManagerError(Exception):
    """Raised for file I/O problems the caller should report to the user."""


class ExpenseFileManager:
    def __init__(self, data_file=os.path.join("data", "expenses.csv"),
                 backup_dir="backups", reports_dir="reports"):
        self.data_file = data_file
        self.backup_dir = backup_dir
        self.reports_dir = reports_dir
        self._ensure_paths()

    def _ensure_paths(self):
        try:
            os.makedirs(os.path.dirname(self.data_file) or ".", exist_ok=True)
            os.makedirs(self.backup_dir, exist_ok=True)
            os.makedirs(self.reports_dir, exist_ok=True)
        except OSError as exc:
            raise FileManagerError(f"Could not create required folders: {exc}")
        if not os.path.exists(self.data_file):
            self.save_expenses([])

    # ------------------------------------------------------------------
    # Load / save
    # ------------------------------------------------------------------
    def load_expenses(self):
        expenses, skipped = [], 0
        if not os.path.exists(self.data_file):
            return expenses
        try:
            with open(self.data_file, newline="", encoding="utf-8") as f:
                reader = csv.reader(f)
                next(reader, None)  # header
                for row in reader:
                    if not row or not any(cell.strip() for cell in row):
                        continue
                    try:
                        expenses.append(Expense.from_row(row))
                    except (ExpenseValidationError, ValueError):
                        skipped += 1
        except (OSError, csv.Error, UnicodeDecodeError) as exc:
            raise FileManagerError(f"Failed to read '{self.data_file}': {exc}")

        if skipped:
            print(f"⚠ Warning: skipped {skipped} malformed row(s) in {self.data_file}.")

        # Duplicate IDs would make update/delete ambiguous; re-number repeats.
        seen = set()
        next_id = max((e.id for e in expenses), default=0) + 1
        for e in expenses:
            if e.id in seen:
                e.id = next_id
                next_id += 1
            seen.add(e.id)
        Expense.reset_id_counter(next_id)
        return expenses

    def save_expenses(self, expenses):
        """Write atomically (temp file, then replace) so a crash mid-write
        can never leave a half-written CSV."""
        tmp_path = self.data_file + ".tmp"
        try:
            self._write_csv(tmp_path, expenses)
            os.replace(tmp_path, self.data_file)
        except OSError as exc:
            raise FileManagerError(f"Failed to write '{self.data_file}': {exc}")

    @staticmethod
    def _write_csv(path, expenses):
        with open(path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(HEADER)
            for e in sorted(expenses, key=lambda x: x.id):
                writer.writerow(e.to_row())

    # ------------------------------------------------------------------
    # Backup / restore
    # ------------------------------------------------------------------
    def create_backup(self):
        if not os.path.exists(self.data_file):
            raise FileManagerError("No data file exists yet, nothing to back up.")
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        base = os.path.splitext(os.path.basename(self.data_file))[0]
        path = os.path.join(self.backup_dir, f"{base}_backup_{stamp}.csv")
        try:
            shutil.copy2(self.data_file, path)
        except OSError as exc:
            raise FileManagerError(f"Backup failed: {exc}")
        return path

    def list_backups(self):
        """Backup file names, newest first."""
        if not os.path.isdir(self.backup_dir):
            return []
        return sorted((f for f in os.listdir(self.backup_dir) if f.endswith(".csv")),
                      reverse=True)

    def restore_backup(self, backup_filename):
        """Restore from a backup. The current data is backed up first so a
        restore can always be undone."""
        # Only plain file names are allowed (blocks '../' tricks).
        if os.path.basename(backup_filename) != backup_filename:
            raise FileManagerError("Invalid backup name.")
        backup_path = os.path.join(self.backup_dir, backup_filename)
        if not os.path.isfile(backup_path):
            raise FileManagerError(f"Backup '{backup_filename}' not found.")
        if os.path.exists(self.data_file):
            self.create_backup()
        try:
            shutil.copy2(backup_path, self.data_file)
        except OSError as exc:
            raise FileManagerError(f"Restore failed: {exc}")

    # ------------------------------------------------------------------
    # Exports
    # ------------------------------------------------------------------
    def export_report(self, text, prefix="report"):
        try:
            os.makedirs(self.reports_dir, exist_ok=True)
            stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            path = os.path.join(self.reports_dir, f"{prefix}_{stamp}.txt")
            with open(path, "w", encoding="utf-8") as f:
                f.write(text)
        except OSError as exc:
            raise FileManagerError(f"Failed to write report: {exc}")
        return path

    def export_csv_copy(self, expenses, path):
        try:
            os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
            self._write_csv(path, expenses)
        except OSError as exc:
            raise FileManagerError(f"Failed to export CSV to '{path}': {exc}")
        return path


Writing personal_finance_manager/file_manager.py


## reports.py

In [15]:
%%writefile "personal_finance_manager/reports.py"
"""
reports.py
----------
Calculations and report text built from a list of Expense objects.
Pure functions: no input/output, so they are easy to test and reuse.
"""

from collections import defaultdict
from datetime import datetime
from statistics import mean, median

from utils import format_currency


# ----------------------------------------------------------------------
# Calculations
# ----------------------------------------------------------------------

def total_spent(expenses):
    return round(sum(e.amount for e in expenses), 2)


def average_spent(expenses):
    return round(mean(e.amount for e in expenses), 2) if expenses else 0.0


def median_spent(expenses):
    return round(median(e.amount for e in expenses), 2) if expenses else 0.0


def category_breakdown(expenses):
    """{category: total}, highest spend first."""
    totals = defaultdict(float)
    for e in expenses:
        totals[e.category] += e.amount
    return {k: round(v, 2) for k, v in
            sorted(totals.items(), key=lambda kv: kv[1], reverse=True)}


def category_counts(expenses):
    counts = defaultdict(int)
    for e in expenses:
        counts[e.category] += 1
    return dict(counts)


def monthly_breakdown(expenses):
    """{'YYYY-MM': total}, oldest month first."""
    totals = defaultdict(float)
    for e in expenses:
        totals[e.date[:7]] += e.amount
    return {k: round(v, 2) for k, v in sorted(totals.items())}


def top_expenses(expenses, n=5):
    return sorted(expenses, key=lambda e: e.amount, reverse=True)[:n]


def filter_by_category(expenses, category):
    category = category.strip().title()
    return [e for e in expenses if e.category == category]


def filter_by_date_range(expenses, start_date=None, end_date=None):
    """start_date / end_date are 'YYYY-MM-DD' strings (inclusive) or None."""
    return [e for e in expenses
            if (start_date is None or e.date >= start_date)
            and (end_date is None or e.date <= end_date)]


def filter_by_month(expenses, month):
    """month is 'YYYY-MM'."""
    return [e for e in expenses if e.date.startswith(month)]


def spending_trend(expenses):
    """Latest month vs the one before it, or None if fewer than two months."""
    monthly = monthly_breakdown(expenses)
    months = list(monthly)
    if len(months) < 2:
        return None
    prev_m, curr_m = months[-2], months[-1]
    prev_t, curr_t = monthly[prev_m], monthly[curr_m]
    pct = round((curr_t - prev_t) / prev_t * 100, 1) if prev_t else None
    return {"previous_month": prev_m, "previous_total": prev_t,
            "current_month": curr_m, "current_total": curr_t, "pct_change": pct}


# ----------------------------------------------------------------------
# Report text
# ----------------------------------------------------------------------

def _category_lines(expenses, width_name=22):
    breakdown = category_breakdown(expenses)
    counts = category_counts(expenses)
    grand_total = total_spent(expenses)
    max_amount = max(breakdown.values()) if breakdown else 1
    lines = []
    for cat, amt in breakdown.items():
        pct = (amt / grand_total * 100) if grand_total else 0
        bar = "#" * int(amt / max_amount * 20)
        lines.append(f"{cat:<{width_name}} {format_currency(amt):>14} ({pct:5.1f}%) "
                     f"[{counts[cat]:>3}] {bar}")
    return lines


def build_summary_report(expenses, title="PERSONAL FINANCE SUMMARY REPORT"):
    width = 66
    lines = ["=" * width, title.center(width),
             f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}".center(width),
             "=" * width]
    if not expenses:
        lines.append("\nNo expenses recorded yet.")
        return "\n".join(lines)

    largest = max(expenses, key=lambda e: e.amount)
    smallest = min(expenses, key=lambda e: e.amount)
    lines += [
        f"\nTotal expenses recorded : {len(expenses)}",
        f"Total amount spent      : {format_currency(total_spent(expenses))}",
        f"Average per expense     : {format_currency(average_spent(expenses))}",
        f"Median expense          : {format_currency(median_spent(expenses))}",
        f"Largest expense         : {format_currency(largest.amount)} "
        f"({largest.category}, {largest.date})",
        f"Smallest expense        : {format_currency(smallest.amount)} "
        f"({smallest.category}, {smallest.date})",
        "\n" + "-" * width, "SPENDING BY CATEGORY", "-" * width,
    ]
    lines += _category_lines(expenses)

    lines += ["\n" + "-" * width, "SPENDING BY MONTH", "-" * width]
    for month, amt in monthly_breakdown(expenses).items():
        lines.append(f"{month:<22} {format_currency(amt):>14}")

    trend = spending_trend(expenses)
    if trend:
        pct = trend["pct_change"]
        if pct is None:
            change, direction = "N/A", "n/a"
        else:
            change = f"{pct:+.1f}%"
            direction = "up" if pct > 0 else "down" if pct < 0 else "unchanged"
        lines += ["\n" + "-" * width, "MONTH-OVER-MONTH TREND", "-" * width,
                  f"{trend['previous_month']} -> {trend['current_month']}: "
                  f"{format_currency(trend['previous_total'])} -> "
                  f"{format_currency(trend['current_total'])} ({change}, {direction})"]

    lines += ["\n" + "-" * width, "TOP 5 LARGEST EXPENSES", "-" * width]
    lines += [str(e) for e in top_expenses(expenses, 5)]
    lines.append("\n" + "=" * width)
    return "\n".join(lines)


def build_monthly_report(expenses, month):
    """Report for a single 'YYYY-MM' month."""
    subset = filter_by_month(expenses, month)
    return build_summary_report(subset, title=f"MONTHLY REPORT: {month}")


Writing personal_finance_manager/reports.py


## menu.py

In [16]:
%%writefile "personal_finance_manager/menu.py"
"""
menu.py
-------
Interactive command-line interface. Owns the ExpenseManager (in-memory
state) and the menu loop. Persistence lives in file_manager.py,
calculations in reports.py, validation in utils.py.
"""

import reports
import utils
from expense import CATEGORIES, Expense, ExpenseValidationError
from file_manager import FileManagerError


class ExpenseManager:
    """In-memory list of expenses, kept in sync with disk via the file manager."""

    def __init__(self, file_manager):
        self.file_manager = file_manager
        self.expenses = []

    def load(self):
        try:
            self.expenses = self.file_manager.load_expenses()
        except FileManagerError as exc:
            utils.print_error(f"Could not load data: {exc}")
            self.expenses = []

    def save(self):
        try:
            self.file_manager.save_expenses(self.expenses)
        except FileManagerError as exc:
            utils.print_error(f"Could not save data: {exc}")
            return False
        return True

    def add_expense(self, expense):
        self.expenses.append(expense)
        return self.save()

    def delete_expense(self, expense_id):
        target = self.find_by_id(expense_id)
        if target is None:
            return False
        self.expenses.remove(target)
        if not self.save():
            self.expenses.append(target)  # keep memory and disk consistent
            return False
        return True

    def update_expense(self, expense_id, **changes):
        """Returns the updated Expense, or None if not found / not saved."""
        target = self.find_by_id(expense_id)
        if target is None:
            return None
        old = (target.amount, target.category, target.date, target.description)
        target.update(**changes)  # raises ExpenseValidationError on bad input
        if not self.save():
            target.amount, target.category, target.date, target.description = old
            return None
        return target

    def find_by_id(self, expense_id):
        return next((e for e in self.expenses if e.id == expense_id), None)


# ----------------------------------------------------------------------
# Screen helpers
# ----------------------------------------------------------------------

APP_TITLE = "PERSONAL FINANCE MANAGER"

MAIN_MENU_TEXT = """
MAIN MENU:
1. Add New Expense
2. View All Expenses
3. View Category-wise Summary
4. Generate Monthly Report
5. Search Expenses
6. Backup Data
7. Exit
"""


def print_banner():
    print("=" * 42)
    print(APP_TITLE.center(42))
    print("=" * 42)


def pause():
    input("\nPress Enter to continue...")


def run_app(manager):
    print_banner()
    print(f"\nLoaded {len(manager.expenses)} existing expense(s) from "
          f"'{manager.file_manager.data_file}'.")

    handlers = {
        1: handle_add_expense,
        2: handle_view_all,
        3: handle_category_summary,
        4: handle_monthly_report,
        5: handle_search,
        6: handle_backup_menu,
    }
    while True:
        print(MAIN_MENU_TEXT)
        choice = utils.prompt_int_in_range("Enter your choice (1-7): ", 1, 7)
        if choice == 7:
            manager.save()
            print("\n✅ Goodbye! Your data has been saved.")
            break
        handlers[choice](manager)


# ----------------------------------------------------------------------
# 1. Add
# ----------------------------------------------------------------------

def handle_add_expense(manager):
    print("\nADD NEW EXPENSE:  (type 'c' at any prompt to cancel)")

    amount = utils.prompt_with_validation("Enter amount: ", utils.parse_amount)
    if amount is None:
        return _cancelled()

    category = utils.prompt_with_validation(
        f"Enter category ({utils.category_hint()}): ", utils.parse_category)
    if category is None:
        return _cancelled()

    date = utils.prompt_with_validation(
        "Enter date (YYYY-MM-DD, blank = today): ", utils.parse_date)
    if date is None:
        return _cancelled()

    while True:
        raw = input("Enter description: ")
        try:
            description = utils.parse_description(raw)
            break
        except ValueError as exc:
            utils.print_error(str(exc))

    try:
        expense = Expense(amount=amount, category=category, date=date,
                          description=description)
    except ExpenseValidationError as exc:
        utils.print_error(f"Could not add expense: {exc}")
        return pause()

    if manager.add_expense(expense):
        print("\n✅ Expense added successfully!")
    else:
        utils.print_error("Expense is in memory but could not be saved to disk.")
    pause()


def _cancelled():
    utils.print_warning("Cancelled.")
    pause()


# ----------------------------------------------------------------------
# 2. View all (+ update / delete)
# ----------------------------------------------------------------------

def handle_view_all(manager):
    print("\nALL EXPENSES:")
    if not manager.expenses:
        utils.print_warning("No expenses recorded yet.")
        return pause()

    items = sorted(manager.expenses)
    for e in items:
        print(f"[{e.id}] {e}")
    print(f"\nTotal: {len(items)} expense(s), "
          f"{utils.format_currency(reports.total_spent(items))}")

    print("\nOptions: [u] Update an expense  [d] Delete an expense  [Enter] Back to menu")
    action = input("Choice: ").strip().lower()
    if action == "u":
        _update_expense_flow(manager)
    elif action == "d":
        _delete_expense_flow(manager)


def _ask_expense_id(manager, verb):
    raw = input(f"Enter the ID of the expense to {verb}: ").strip()
    try:
        expense_id = int(raw)
    except ValueError:
        utils.print_error("Please enter a valid numeric ID.")
        return None
    existing = manager.find_by_id(expense_id)
    if existing is None:
        utils.print_error(f"No expense found with ID {expense_id}.")
    return existing


def _update_expense_flow(manager):
    existing = _ask_expense_id(manager, "update")
    if existing is None:
        return pause()

    print(f"Current: {existing}")
    print("(Press Enter to keep a value, 'c' to cancel)")

    new_amount = utils.prompt_with_validation(
        f"New amount [{existing.amount}]: ", utils.parse_amount, default=existing.amount)
    if new_amount is None:
        return _cancelled()
    new_category = utils.prompt_with_validation(
        f"New category [{existing.category}]: ", utils.parse_category,
        default=existing.category)
    if new_category is None:
        return _cancelled()
    new_date = utils.prompt_with_validation(
        f"New date [{existing.date}]: ",
        lambda r: utils.parse_date(r, allow_empty_today=False), default=existing.date)
    if new_date is None:
        return _cancelled()

    while True:
        raw = input(f"New description [{existing.description}]: ").strip()
        try:
            new_description = utils.parse_description(raw) if raw else existing.description
            break
        except ValueError as exc:
            utils.print_error(str(exc))

    try:
        updated = manager.update_expense(
            existing.id, amount=new_amount, category=new_category,
            date=new_date, description=new_description)
    except ExpenseValidationError as exc:
        utils.print_error(f"Update failed: {exc}")
        return pause()

    if updated:
        print(f"\n✅ Expense updated: {updated}")
    else:
        utils.print_error("Update could not be saved.")
    pause()


def _delete_expense_flow(manager):
    existing = _ask_expense_id(manager, "delete")
    if existing is None:
        return pause()

    print(f"About to delete: {existing}")
    if not utils.confirm("Confirm deletion? (y/n): "):
        return _cancelled()

    if manager.delete_expense(existing.id):
        print(f"\n✅ Expense {existing.id} deleted successfully!")
    else:
        utils.print_error("Delete could not be saved.")
    pause()


# ----------------------------------------------------------------------
# 3. Category summary
# ----------------------------------------------------------------------

def handle_category_summary(manager):
    print("\nCATEGORY-WISE SUMMARY:")
    if not manager.expenses:
        utils.print_warning("No expenses recorded yet.")
        return pause()

    breakdown = reports.category_breakdown(manager.expenses)
    counts = reports.category_counts(manager.expenses)
    grand_total = reports.total_spent(manager.expenses)
    for cat, amt in breakdown.items():
        pct = (amt / grand_total * 100) if grand_total else 0
        print(f"{cat:<22} {utils.format_currency(amt):>14}  "
              f"({pct:5.1f}%)  [{counts[cat]} item(s)]")
    print(f"\nGrand Total: {utils.format_currency(grand_total)}")
    pause()


# ----------------------------------------------------------------------
# 4. Monthly report
# ----------------------------------------------------------------------

def _parse_month(raw):
    from datetime import datetime
    raw = raw.strip()
    try:
        return datetime.strptime(raw, "%Y-%m").strftime("%Y-%m")
    except ValueError:
        raise ValueError(f"'{raw}' is not a valid month. Use YYYY-MM (e.g. 2024-01).")


def handle_monthly_report(manager):
    print("\nMONTHLY REPORT:")
    if not manager.expenses:
        utils.print_warning("No expenses recorded yet.")
        return pause()

    months = list(reports.monthly_breakdown(manager.expenses))
    print(f"Months with data: {', '.join(months)}")
    month = utils.prompt_with_validation(
        "Month to report on (YYYY-MM, blank = all time): ",
        lambda r: _parse_month(r) if r.strip() else "")
    if month is None:
        return _cancelled()

    if month:
        if month not in months:
            utils.print_warning(f"No expenses recorded in {month}.")
            return pause()
        report_text = reports.build_monthly_report(manager.expenses, month)
        prefix = f"report_{month}"
    else:
        report_text = reports.build_summary_report(manager.expenses)
        prefix = "report_all"
    print(report_text)

    if utils.confirm("\nSave this report to a file? (y/n): "):
        try:
            path = manager.file_manager.export_report(report_text, prefix=prefix)
            print(f"\n✅ Report saved to '{path}'.")
        except FileManagerError as exc:
            utils.print_error(str(exc))
    pause()


# ----------------------------------------------------------------------
# 5. Search
# ----------------------------------------------------------------------

def handle_search(manager):
    print("\nSEARCH EXPENSES:")
    print("1. By category")
    print("2. By date range")
    print("3. By keyword in description")
    choice = utils.prompt_int_in_range("Select an option (1-3): ", 1, 3)

    if choice == 1:
        category = utils.prompt_nonempty("Category to search for: ")
        results = reports.filter_by_category(manager.expenses, category)
    elif choice == 2:
        def parse_bound(raw):
            return utils.parse_date(raw, allow_empty_today=False) if raw.strip() else ""
        start = utils.prompt_with_validation(
            "Start date (YYYY-MM-DD, blank = no lower bound): ", parse_bound)
        if start is None:
            return _cancelled()
        end = utils.prompt_with_validation(
            "End date (YYYY-MM-DD, blank = no upper bound): ", parse_bound)
        if end is None:
            return _cancelled()
        start, end = start or None, end or None
        if start and end and start > end:
            utils.print_error("Start date is after the end date.")
            return pause()
        results = reports.filter_by_date_range(manager.expenses, start, end)
    else:
        keyword = utils.prompt_nonempty("Keyword to search for: ").lower()
        results = [e for e in manager.expenses if keyword in e.description.lower()]

    if not results:
        utils.print_warning("No matching expenses found.")
        return pause()

    for e in sorted(results):
        print(f"[{e.id}] {e}")
    print(f"\nFound {len(results)} expense(s), total "
          f"{utils.format_currency(reports.total_spent(results))}")
    pause()


# ----------------------------------------------------------------------
# 6. Backup / restore / export
# ----------------------------------------------------------------------

def handle_backup_menu(manager):
    print("\nBACKUP DATA:")
    print("1. Create backup now")
    print("2. List backups")
    print("3. Restore from backup")
    print("4. Export expenses to a CSV file")
    choice = utils.prompt_int_in_range("Select an option (1-4): ", 1, 4)
    fm = manager.file_manager

    if choice == 1:
        try:
            manager.save()
            path = fm.create_backup()
            print(f"\n✅ Backup created successfully: '{path}'")
        except FileManagerError as exc:
            utils.print_error(str(exc))

    elif choice == 2:
        backups = fm.list_backups()
        if not backups:
            utils.print_warning("No backups found.")
        for i, name in enumerate(backups, 1):
            print(f" {i}. {name}")

    elif choice == 3:
        backups = fm.list_backups()
        if not backups:
            utils.print_warning("No backups available to restore.")
            return pause()
        for i, name in enumerate(backups, 1):
            print(f" {i}. {name}")
        idx = utils.prompt_int_in_range(
            f"Select a backup to restore (1-{len(backups)}): ", 1, len(backups))
        chosen = backups[idx - 1]
        if not utils.confirm(
                f"This will replace current data with '{chosen}' "
                f"(current data is backed up first). Continue? (y/n): "):
            return _cancelled()
        try:
            fm.restore_backup(chosen)
            manager.load()
            print(f"\n✅ Data restored from '{chosen}'. "
                  f"{len(manager.expenses)} expense(s) loaded.")
        except FileManagerError as exc:
            utils.print_error(str(exc))

    else:
        path = input("Export file path (e.g. reports/export.csv): ").strip()
        if not path:
            return _cancelled()
        try:
            exported = fm.export_csv_copy(manager.expenses, path)
            print(f"\n✅ Exported {len(manager.expenses)} expense(s) to '{exported}'.")
        except FileManagerError as exc:
            utils.print_error(str(exc))

    pause()


Writing personal_finance_manager/menu.py


## main.py

In [17]:
%%writefile "personal_finance_manager/main.py"
"""
main.py
-------
Entry point for the Personal Finance Manager.

Run with:
    python main.py

Kept thin on purpose: it wires the file manager and expense manager
together and hands control to the menu loop.
"""

import os
import sys

# Always keep data next to the program, no matter where it is launched from.
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
os.chdir(BASE_DIR)

from file_manager import ExpenseFileManager, FileManagerError  # noqa: E402
from menu import ExpenseManager, run_app                       # noqa: E402
from utils import print_error                                  # noqa: E402

DATA_FILE = os.path.join("data", "expenses.csv")
BACKUP_DIR = "backups"
REPORTS_DIR = "reports"


def main():
    try:
        file_manager = ExpenseFileManager(DATA_FILE, BACKUP_DIR, REPORTS_DIR)
    except FileManagerError as exc:
        print_error(f"Fatal error setting up data storage: {exc}")
        sys.exit(1)

    manager = ExpenseManager(file_manager)
    manager.load()

    try:
        run_app(manager)
    except (KeyboardInterrupt, EOFError):
        print("\n\nInterrupted. Saving data before exit...")
        manager.save()
        print("Goodbye!")
        sys.exit(0)
    except Exception as exc:  # last-resort safety net so data is never lost
        print_error(f"An unexpected error occurred: {exc}")
        manager.save()
        print("Your data has been saved. Please report this issue.")
        sys.exit(1)


if __name__ == "__main__":
    main()


Writing personal_finance_manager/main.py


## Sample data (optional)
Delete the rows below the header if you want to start with an empty tracker.

In [18]:
%%writefile "personal_finance_manager/data/expenses.csv"
ID,Date,Category,Amount,Description
1,2024-01-05,Food,1500.00,Grocery shopping
2,2024-01-12,Transport,450.00,Metro card recharge
3,2024-01-20,Entertainment,800.00,Movie night
4,2024-02-03,Shopping,3200.00,New shoes
5,2024-02-10,Food,1750.50,Weekly groceries
6,2024-02-18,Bills,2100.00,Electricity bill
7,2024-03-02,Health,950.00,Pharmacy
8,2024-03-15,Food,1620.00,Grocery shopping


Writing personal_finance_manager/data/expenses.csv


## Run the app
Type your choices in the box that appears under the cell. Choose **7** to exit.

In [22]:
import sys

%cd personal_finance_manager

# Drop cached copies so edits made in the cells above take effect on re-run
for m in ["expense", "utils", "file_manager", "reports", "menu", "main"]:
    sys.modules.pop(m, None)

%run main.py

[Errno 2] No such file or directory: 'personal_finance_manager'
/content/personal_finance_manager/personal_finance_manager
         PERSONAL FINANCE MANAGER         

Loaded 9 existing expense(s) from 'data/expenses.csv'.

MAIN MENU:
1. Add New Expense
2. View All Expenses
3. View Category-wise Summary
4. Generate Monthly Report
5. Search Expenses
6. Backup Data
7. Exit

Enter your choice (1-7): 1

ADD NEW EXPENSE:  (type 'c' at any prompt to cancel)
Enter amount: 999
Enter category (Food/Transport/Entertainment/Shopping/Bills/Health/Education/Other): Other
Enter date (YYYY-MM-DD, blank = today): 
Enter description: Intership

✅ Expense added successfully!

Press Enter to continue...

MAIN MENU:
1. Add New Expense
2. View All Expenses
3. View Category-wise Summary
4. Generate Monthly Report
5. Search Expenses
6. Backup Data
7. Exit

Enter your choice (1-7): 5

SEARCH EXPENSES:
1. By category
2. By date range
3. By keyword in description
Select an option (1-3): internship
❌ Please enter